In [1]:
import numpy as np
import pandas as pd

# --- 1. LFP Benchmark Specifications (Locked) ---
pack_specs = {
    'V_nom': 51.2, # V
    'Cap_nom': 100, # Ah
    'Energy': 5120, # Wh
    'm_pack': 48, # kg
    'cp': 950, # J/kgK
    'R_th_nominal': 0.35, # K/W
    'T_cool_limit': 38.0, # Celsius
    'T_ideal': 25.0, # Celsius
    'A_surface': 1.2 # m^2 (estimated for 5kWh rack)
}

# Derived Thermal Capacitance
C_th = pack_specs['m_pack'] * pack_specs['cp']
print(f'Thermal Capacitance (C_th): {C_th} J/K')

# --- 2. Climate & Passive Heat Transfer (h) ---
# h = P_heat / (A * delta_T)
# Given target h values from specs
h_coastal = 6.5 # W/m^2K
h_harmattan = 8.8 # W/m^2K

def calculate_steady_state_delta_T(P_heat, h, A):
    return P_heat / (h * A)

# --- 3. Grid Disturbance Calibration (w_grid,env) ---
def calculate_grid_stress():
    # Empirical Weights
    alpha = 0.5 # Voltage deviation (%)
    beta = 2.0  # Freq deviation (Hz)
    gamma = 10.0 # Blackout penalty

    # Data Inputs
    dv = 11.0 # 11% deviation
    df = 0.97 # Hz deviation

    # D_k calculation
    D_collapse = alpha*dv + beta*df + gamma*1
    D_disturb = alpha*dv + beta*df + gamma*0

    # Frequency of events
    n_blackouts = 350
    n_disturbances = 700
    N = n_blackouts + n_disturbances

    w_grid_env = (n_blackouts * D_collapse + n_disturbances * D_disturb) / N
    return w_grid_env, D_collapse, D_disturb

w_grid_env, d_col, d_dist = calculate_grid_stress()

# --- 4. Thermal Stress Coefficient (w_thermal,env) ---
# Using a representative temperature profile simulation over 24h
def simulate_thermal_coefficient(h_val):
    # Simplified: Average T elevation above T_ideal under 1kW load
    P_avg = 1000 * (0.02) # Assuming 2% avg loss (Joule heating)
    delta_T_avg = P_avg * pack_specs['R_th_nominal']
    T_avg = 32.0 + delta_T_avg # 32C as base ambient
    w_thermal = (T_avg - pack_specs['T_ideal']) / pack_specs['T_ideal']
    return w_thermal

w_therm_coastal = simulate_thermal_coefficient(h_coastal)

# --- Results Output ---
results = {
    'Parameter': ['R_th', 'h_coastal', 'h_harmattan', 'T_cool_LFP', 'w_grid_env', 'w_thermal_env'],
    'Value': [pack_specs['R_th_nominal'], h_coastal, h_harmattan, pack_specs['T_cool_limit'], round(w_grid_env, 2), round(w_therm_coastal, 3)],
    'Unit': ['K/W', 'W/m^2K', 'W/m^2K', '°C', 'dimensionless', 'dimensionless']
}

df_results = pd.DataFrame(results)
display(df_results)

Thermal Capacitance (C_th): 45600 J/K


,Parameter,Value,Unit
0,R_th,0.35,K/W
1,h_coastal,6.50,W/m^2K
2,h_harmattan,8.80,W/m^2K
3,T_cool_LFP,38.00,°C
4,w_grid_env,10.77,dimensionless
5,w_thermal_env,0.56,dimensionless


In [2]:
def calculate_hotspot_sensitivity(P1, P2, T1, T2):
    """
    S_hotspot = delta_T / delta_P
    """
    return (T2 - T1) / (P2 - P1)

# Typical Pulse response for LFP anchor
s_hotspot = calculate_hotspot_sensitivity(100, 200, 35.5, 38.2)
print(f'Calculated Hotspot Sensitivity (S_hotspot): {s_hotspot:.4f} K/W')

Calculated Hotspot Sensitivity (S_hotspot): 0.0270 K/W


In [3]:
import pandas as pd
import numpy as np

# --- 1. Inputs: Locked LFP Benchmark & Nigerian Grid Data ---
LFP_SPECS = {
    'm_pack': 48,           # kg
    'cp': 950,              # J/kgK
    'A_surface': 1.2,       # m^2
    'T_ideal': 25.0,        # °C
    'T_cool_LFP': 38.0,     # °C
    'P_avg_loss': 20.0,     # W (Assumed avg heat dissipation during cycle)
    'delta_T_target': 7.0   # K (Target steady state rise for R_th calibration)
}

# --- 2. Thermal Network Parameters ---
# R_th = delta_T / P_heat
R_th = LFP_SPECS['delta_T_target'] / LFP_SPECS['P_avg_loss']

# Passive heat transfer coefficients (h = P / (A * delta_T))
# Calculated for Coastal (Target delta_T ~ 2.56K at 20W loss)
h_coastal = LFP_SPECS['P_avg_loss'] / (LFP_SPECS['A_surface'] * 2.564)

# Calculated for Harmattan (Target delta_T ~ 1.89K at 20W loss)
h_harmattan = LFP_SPECS['P_avg_loss'] / (LFP_SPECS['A_surface'] * 1.894)

# --- 3. Hotspot Sensitivity (S_hotspot) ---
# S_hotspot = (T2 - T1) / (P2 - P1)
def calc_s_hotspot():
    p1, p2 = 100.0, 200.0 # Watts
    t1, t2 = 35.5, 38.2 # Celsius (Representative pulse response)
    return (t2 - t1) / (p2 - p1)

S_hotspot = calc_s_hotspot()

# --- 4. Grid Disturbance Coefficient (w_grid,env) ---
def calc_w_grid():
    # D_k = alpha|dV| + beta|df| + gamma*B
    alpha, beta, gamma = 0.5, 2.0, 10.0
    dv, df = 11.0, 0.97

    D_collapse = alpha*dv + beta*df + (gamma * 1)
    D_disturb = alpha*dv + beta*df + (gamma * 0)

    n_blackouts = 350
    n_disturbances = 700
    N = n_blackouts + n_disturbances

    return (n_blackouts * D_collapse + n_disturbances * D_disturb) / N

w_grid_env = calc_w_grid()

# --- 5. Thermal Stress Coefficient (w_thermal,env) ---
def calc_w_thermal():
    # w = (T_avg_op - T_ideal) / T_ideal
    T_avg_op = 32.0 + (LFP_SPECS['P_avg_loss'] * R_th)
    return (T_avg_op - LFP_SPECS['T_ideal']) / LFP_SPECS['T_ideal']

w_thermal_env = calc_w_thermal()

# --- Final Phase 0 Output Table ---
phase0_summary = {
    "Calibration Output": [
        "Effective Thermal Resistance (R_th)",
        "Heat Transfer Coeff (h_coastal)",
        "Heat Transfer Coeff (h_harmattan)",
        "Cooling Threshold (T_cool,LFP)",
        "Hotspot Sensitivity (S_hotspot)",
        "Thermal Stress Coeff (w_thermal,env)",
        "Grid Disturbance Coeff (w_grid,env)"
    ],
    "Value": [
        round(R_th, 3),
        round(h_coastal, 2),
        round(h_harmattan, 2),
        LFP_SPECS['T_cool_LFP'],
        round(S_hotspot, 4),
        round(w_thermal_env, 3),
        round(w_grid_env, 2)
    ],
    "Unit": [
        "K/W", "W/m^2K", "W/m^2K", "°C", "K/W", "dimensionless", "dimensionless"
    ]
}

df_phase0 = pd.DataFrame(phase0_summary)
print("PHASE 0: ENVIRONMENTAL MODEL CALIBRATION (LFP ANCHOR) COMPLETED")
display(df_phase0)

PHASE 0: ENVIRONMENTAL MODEL CALIBRATION (LFP ANCHOR) COMPLETED


,Calibration Output,Value,Unit
0,Effective Thermal Resistance (R_th),0.350,K/W
1,Heat Transfer Coeff (h_coastal),6.500,W/m^2K
2,Heat Transfer Coeff (h_harmattan),8.800,W/m^2K
3,"Cooling Threshold (T_cool,LFP)",38.000,°C
4,Hotspot Sensitivity (S_hotspot),0.027,K/W
5,"Thermal Stress Coeff (w_thermal,env)",0.560,dimensionless
6,"Grid Disturbance Coeff (w_grid,env)",10.770,dimensionless
